# 01 CAPMと市場リスク

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/factors.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
市場感応度βと切片αを推定し、真の値と推定誤差を比較する。

## 2. 理論と数式
$r_i-r_f=\alpha+\beta(r_m-r_f)+\epsilon$。OLSとHAC標準誤差。αは日次。

## 3. データの定義
seed42、750営業日、真のβ1.35・α0.0001の合成日次リターン。実市場ではない。

## 4. 分析条件・パラメータ
rf日次=(1.02)^(1/252)-1、HACラグは関数既定5。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/quant_ai").exists())
os.chdir(ROOT)
(ROOT / "分析結果/図").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("データ種別: 合成データ。実市場での収益性を示す結果ではありません。")


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.factors import capm_beta
print(inspect.getsourcefile(capm_beta))
display(Code(inspect.getsource(capm_beta), language='python'))

In [ ]:
from quant_ai.factors import capm_beta
rng = np.random.default_rng(42)
dates = pd.bdate_range("2020-01-01", periods=750)
market = pd.Series(rng.normal(0.0003, 0.01, len(dates)), index=dates)
rf_daily = (1.02 ** (1/252) - 1)
asset = rf_daily + 0.0001 + 1.35*(market-rf_daily) + rng.normal(0, 0.006, len(dates))
model = capm_beta(asset, market, risk_free_daily=rf_daily)
display(model.summary())
display(model.conf_int().rename(columns={0: "lower", 1: "upper"}))
print("真のβ: 1.35 / 推定β:", model.params["market_excess"])


## 6. 計算結果と可視化

In [ ]:
x, y = market-rf_daily, asset-rf_daily
fig, ax = plt.subplots()
ax.scatter(x, y, s=9, alpha=0.3)
grid = np.linspace(x.min(), x.max(), 100)
ax.plot(grid, model.params["const"] + model.params["market_excess"]*grid, color="red")
ax.set(xlabel="Market excess return (daily)", ylabel="Asset excess return (daily)",
       title="CAPM regression - synthetic data")
fig.savefig("分析結果/図/01_CAPM回帰.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 統計的解釈
信頼区間・HAC標準誤差・R²を確認する。βの有意性は予測力を意味しない。

## 8. 金融工学・運用上の解釈
βは市場超過収益への感応度であり、期待超過収益の保証ではない。

## 9. 限界と追加検証
## 解釈
推定βと1.35の差は有限標本の誤差です。αの信頼区間が0を含むか確認してください。
高いR²や有意なβは予測可能性を意味しません。これは同時点の説明回帰です。

## 限界と改善
単一ファクター、定数β、取引摩擦なしが数理上の仮定です。実市場では市場指数は
市場ポートフォリオの代理変数にすぎず、配当調整、無リスク金利、通貨の違いが影響します。
移動窓β、期間別安定性、多因子回帰、HACラグ感度を比較しましょう。